# Exercise 2.

In [1]:
# --- Read the BLOSUM62 matrix from .txt file ---

# Read and store the whole blosum62.txt file as a list of strings, each string representing a row

f = open("blosum62.txt", "r")
full_file_blosum62 = f.readlines()
f.close()

In [2]:
# --- Create score matrix dictionary ---

# --- Separate the residues ---

# Only looking at second line containing the residues
# Filter out the white space and newline at the end, store each letter as individual string

residues = full_file_blosum62[1].split()


# --- Store the matrix values as nested list (list of lists) --- 

# Each row of values in the matrix is stored as a list of value

score_matrix = []

for row_str in full_file_blosum62[2:]:
    
    # Turn row of string into list of string elements
    row_list = row_str.split()

    row_value_list = []

    for elem in row_list[1:]:
        # Skipping first element which is a residue letter

        row_value_list.append(int(elem))

    score_matrix.append(row_value_list)


# --- Turn into dictionary ---

# Since the list of residues have indices corresponding to the position in the scoring matrix we can easily construct a dictionary with
# pairs of residues (as tuple of two string) and their score

blosum62_score_matrix_dict = {}

for i, resA in enumerate(residues):
    for j, resB in enumerate(residues):
        blosum62_score_matrix_dict[(resA, resB)] = score_matrix[i][j]

In [3]:
blosum62_score_matrix_dict.get(("W", "W"))

11

In [4]:
def faa_fasta_reader(full_filename):
    """ Return sequence as string of residues """

    f = open(full_filename, "r")
    full_file = f.readlines()
    f.close()

    seq = ""

    # Skip first line in file
    for row_string in full_file[1:]:
        for res in row_string:
            if res == " " or res == "\n":
                pass

            else:
                seq += res

    return seq

In [5]:
faa_fasta_reader("sequence_Q9SIE0_2.fasta")

'MTNPLNSTAANRSNQPSSDGISDGQITNEEAESLINKKNCSGHKLKEVTDSDTFSDNGKDDSDTKKRFHYHQDQRRMSLTSIVAVESPSSSNAPSRKTIDLGHGSDLIYIQRFLPFQQSWTFFDYLDKHIPWTRPTIRVFGRSCLQPRDTCYVASSGLTALVYSGYRPTSYSWDDFPPLKEILDAIYKVLPGSRFNSLLLNRYKGASDYVAWHADDEKIYGPTPEIASVSFGCERDFVLKKKKDEESSQGKTGDSGPAKKRLKRSSREDQQSLTLKHGSLLVMRGYTQRDWIHSVPKRAKAEGTRINLTFRLVL'

In [6]:
def initial_matrix_setup(seqA, seqB, gap_open_pen, gap_extend_pen):

    if gap_open_pen > 0 or gap_extend_pen > 0:
        raise ValueError("Penalties must strictly be negative")

    h = gap_open_pen
    s = gap_extend_pen

    n_rows = len(seqB) + 2
    n_columns = len(seqA) + 2

    V_matrix = []
    E_matrix = []
    F_matrix = []
    G_matrix = []
    backtrack_matrix = []

    for i in range(n_rows):
        row_list_V = []
        row_list_E = []
        row_list_F = []
        row_list_G = []
        row_list_backtrack = []

        for j in range(n_columns):
            row_list_V.append(0)
            row_list_E.append(0)
            row_list_F.append(0)
            row_list_G.append(0)
            row_list_backtrack.append(0)

        V_matrix.append(row_list_V)
        E_matrix.append(row_list_E)
        F_matrix.append(row_list_F)
        G_matrix.append(row_list_G)
        backtrack_matrix.append(row_list_backtrack)

    V_matrix[0][0] = E_matrix[0][0] = F_matrix[0][0] = G_matrix[0][0] = " "
    V_matrix[0][1] = E_matrix[0][1] = F_matrix[0][1] = G_matrix[0][1] = "-"
    V_matrix[1][0] = E_matrix[1][0] = F_matrix[1][0] = G_matrix[1][0] = "-"
    V_matrix[1][1] = E_matrix[1][1] = F_matrix[1][1] = G_matrix[1][1] = 0

    # Filling in row seq. in first row
    for i in range(len(seqA)):
        V_matrix[0][i+2] = seqA[i]
        E_matrix[0][i+2] = seqA[i]
        F_matrix[0][i+2] = seqA[i]
        G_matrix[0][i+2] = seqA[i]

    # Filling in column seq. in first column
    for j in range(len(seqB)):
        V_matrix[j+2][0] = seqB[j]
        E_matrix[j+2][0] = seqB[j]
        F_matrix[j+2][0] = seqB[j]
        G_matrix[j+2][0] = seqB[j]

    # V matrix
    for i in range(2, n_rows):
        V_matrix[i][1] = h + (i-1)*s

    for j in range(2, n_columns):
        V_matrix[1][j] = h + (j-1)*s

    # E matrix
    for i in range(2, n_rows):
        E_matrix[i][1] = 2*h + i*s

    for j in range(2, n_columns):
        E_matrix[1][j] = -99

    # F matrix
    for i in range(2, n_rows):
        F_matrix[i][1] = -99

    for j in range(2, n_columns):
        F_matrix[1][j] = 2*h + j*s
    
    # G matrix
    for i in range(2, n_rows):
        G_matrix[i][1] = h + (i-1)*s

    for j in range(2, n_columns):
        G_matrix[1][j] = h + (j-1)*s
    
    # Backtrack matrix
    # Fill out the gap row and column, with E and F respectively 
    for i in range (2, n_rows):
        backtrack_matrix[i][1] = "F_ext"
    
    for j in range (2, n_columns):
        backtrack_matrix[1][j] = "E_ext"

    # backtrack_matrix[2][1] = "F_open"
    # backtrack_matrix[1][2] = "E_open"

    return V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix

In [7]:
seqA = "AAGT"
seqB = "AT"

h = -11 # opening pen
s = -1 # extend pens

V_matrix, E_matrix, F_matrix, G_matrix, backtrack_matrix = initial_matrix_setup(seqA, seqB, gap_open_pen=h, gap_extend_pen=s)

In [8]:
print("V:")
for row in V_matrix:
    print(row)

print("------------------------------------------")

print("E:")
for row in E_matrix:
    print(row)

print("------------------------------------------")

print("F:")
for row in F_matrix:
    print(row)

print("------------------------------------------")

print("G:")
for row in G_matrix:
    print(row)

print("------------------------------------------")

print("Backtrack:")
for row in backtrack_matrix:
    print(row)

V:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -12, -13, -14, -15]
['A', -12, 0, 0, 0, 0]
['T', -13, 0, 0, 0, 0]
------------------------------------------
E:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -99, -99, -99, -99]
['A', -24, 0, 0, 0, 0]
['T', -25, 0, 0, 0, 0]
------------------------------------------
F:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -24, -25, -26, -27]
['A', -99, 0, 0, 0, 0]
['T', -99, 0, 0, 0, 0]
------------------------------------------
G:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -12, -13, -14, -15]
['A', -12, 0, 0, 0, 0]
['T', -13, 0, 0, 0, 0]
------------------------------------------
Backtrack:
[0, 0, 0, 0, 0, 0]
[0, 0, 'E_ext', 'E_ext', 'E_ext', 'E_ext']
[0, 'F_ext', 0, 0, 0, 0]
[0, 'F_ext', 0, 0, 0, 0]


In [9]:
for i in range(2, len(seqB)+2):
    for j in range(2, len(seqA)+2):
        
        # if V_matrix[i][0] == V_matrix[0][j]:
        #     match_missmatch = 1
        # else:
        #     match_missmatch = -1

        # G_val = V_matrix[i-1][j-1] + match_missmatch

        G_val = V_matrix[i-1][j-1] + blosum62_score_matrix_dict.get((V_matrix[i][0], V_matrix[0][j]))

        # Max value E
        E_val_ext = E_matrix[i][j-1] + s
        E_val_open = V_matrix[i][j-1] + h + s

        if E_val_ext >= E_val_open:
            E_val = E_val_ext
            E_val_symb = "E_ext"
        else:
            E_val = E_val_open
            E_val_symb = "E_open"

        # Max value F
        F_val_ext = F_matrix[i-1][j] + s
        F_val_open = V_matrix[i-1][j] + h + s

        if F_val_ext >= F_val_open:
            F_val = F_val_ext
            F_val_symb = "F_ext"
        else:
            F_val = F_val_open
            F_val_symb = "F_open"

        if G_val >= E_val and G_val >= F_val:
            V_matrix[i][j] = G_val
            backtrack_matrix[i][j] = "G"
            
        if E_val > G_val and E_val > F_val:
            V_matrix[i][j] = E_val
            backtrack_matrix[i][j] = E_val_symb

        if F_val > G_val and F_val > E_val:
            V_matrix[i][j] = F_val
            backtrack_matrix[i][j] = F_val_symb

        # If ties between gaps, prefer gap in shortest seq.                
        if E_val == F_val and E_val > G_val:
            if len(seqA) > len(seqB):
                V_matrix[i][j] = E_val
                backtrack_matrix[i][j] = E_val_symb

            if len(seqB) > len(seqA):
                V_matrix[i][j] = F_val
                backtrack_matrix[i][j] = F_val_symb

            # If seqs. are same len, prefer column seq.
            else:
                V_matrix[i][j] = E_val
                backtrack_matrix[i][j] = E_val_symb
        

In [12]:
print("V:")
for row in V_matrix:
    print(row)

print("------------------------------------------")

print("E:")
for row in E_matrix:
    print(row)

print("------------------------------------------")

print("F:")
for row in F_matrix:
    print(row)

print("------------------------------------------")

print("G:")
for row in G_matrix:
    print(row)

print("------------------------------------------")

print("Backtrack:")
for row in backtrack_matrix:
    print(row)

V:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -12, -13, -14, -15]
['A', -12, 4, -1, -1, -1]
['T', -13, -1, 4, -1, 4]
------------------------------------------
E:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -99, -99, -99, -99]
['A', -24, 0, 0, 0, 0]
['T', -25, 0, 0, 0, 0]
------------------------------------------
F:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -24, -25, -26, -27]
['A', -99, 0, 0, 0, 0]
['T', -99, 0, 0, 0, 0]
------------------------------------------
G:
[' ', '-', 'A', 'A', 'G', 'T']
['-', 0, -12, -13, -14, -15]
['A', -12, 0, 0, 0, 0]
['T', -13, 0, 0, 0, 0]
------------------------------------------
Backtrack:
[0, 0, 0, 0, 0, 0]
[0, 0, 'E_ext', 'E_ext', 'E_ext', 'E_ext']
[0, 'F_ext', 'G', 'E_ext', 'E_ext', 'E_ext']
[0, 'F_ext', 'F_ext', 'G', 'E_ext', 'G']


In [11]:
alignment_seqA = ""
alignment = ""
alignment_seqB = ""

# Start bottom right corner stop when hitting top left corner (1, 1)
# The "padding" of backtrack_matrix with E and F ensures we will not miss (1,1)

i = len(seqB) + 1
j = len(seqA) + 1

E_gap = "off"   # on/off
F_gap = "off"   # on/off

first_backtrack_direction = backtrack_matrix[i][j]
if first_backtrack_direction == "E_ext":
    E_gap = "on"

if first_backtrack_direction == "F_ext":
    F_gap = "on"

print(E_gap, F_gap)

while i > 1  or j > 1:

    backtrack_direction = backtrack_matrix[i][j]
    print(backtrack_direction)
    print(i, j)

    if backtrack_direction == "G":

        if E_gap == "on":
            alignment_seqA += seqA[j-2]
            alignment += " "
            alignment_seqB += "-"
            j -= 1

        elif F_gap == "on":
            alignment_seqB += seqB[i-2]
            alignment += " "
            alignment_seqA += "-"
            i -= 1

        else:
            alignment_seqA += seqA[j-2]
            alignment += "|"
            alignment_seqB += seqB[i-2]

            i -= 1
            j -= 1

            E_gap = "off"
            F_gap = "off" 

    elif backtrack_direction == "E_open":
            alignment_seqA += seqA[j-2]
            alignment += " "
            alignment_seqB += "-"
            j -= 1

            E_gap = "off"

    elif backtrack_direction == "E_ext":
            alignment_seqA += seqA[j-2]
            alignment += " "
            alignment_seqB += "-"
            j -= 1

            E_gap = "on"

    elif backtrack_direction == "F_open":
            alignment_seqB += seqB[i-2]
            alignment += " "
            alignment_seqA += "-"
            i -= 1

            F_gap = "off"

    elif backtrack_direction == "F_ext":
            alignment_seqB += seqB[i-1]
            alignment += " "
            alignment_seqA += "-"
            i -= 1

            F_gap = "on"

    print(i,j)
    print(alignment_seqA, alignment_seqB)

print(alignment_seqA[::-1])
print(alignment[::-1])
print(alignment_seqB[::-1])

off off
G
3 5
2 4
T T
E_ext
2 4
2 3
TG T-
E_ext
2 3
2 2
TGA T--
G
2 2
2 1
TGAA T---
F_ext
2 1
1 1
TGAA- T---T
-AAGT
    |
T---T
